In [1]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests
from app.utils.db import save_results_on_cosmos

In [4]:
file_list = [
  # './app/data/raw/tramites.xlsx',
  # './app/data/raw/accesibilidad.xlsx',
  # './app/data/raw/descubrir.xlsx', 
  # './app/data/raw/solicitudes.xlsx',
  # './app/data/raw/organigrama.xlsx'
]

test_config = {
  'GENERAL_TESTS': False,
  'TIMINGS': {'test': False, 'report': False},
  'TOKENS': {'test': False, 'report': False},
  'FOUNDRYS': {'test': False, 'report': False},
  'TRIAGE': {'test': False, 'report': False},
  'ROUTER': {'test': False, 'report': False},
  'GROUNDING': {'test': False, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report',
  
  'REFORMULATE': {'test': True, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [5]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = response_file_path

# SOLO REFORMULATE
if test_config.get('REFORMULATE', False).get('test', False):
  reformulate_dataset = load_test_cases('./app/data/raw/reformulate.xlsx')
  reformulate_results = send_reformulate_requests(config= config_data, dataset=reformulate_dataset)
  generate_reformulate_json, reformulate_timestamp = save_reformualte_responses(reformulate_results)
  test_timestamps['reformulate_test'] = reformulate_timestamp
  reformulate_results = reformulate_tests(reformulate_results)

In [6]:
print(reformulate_results)

4.6


In [4]:
import json

response_file_path = './app/data/processed/outcome_20260406-153646.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260406-153646.json'

In [5]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
  )

In [6]:
print(results)

result_one = {'timestamp': '20260330-185244', 'nodes': {'triage': {'positives': 870, 'total': 901, 'result': 96.56}, 'router': {'positives': 805, 'total': 870, 'result': 92.53}}, 'timings': {'reformulate': {'prom': 0.885, 'p90': 1.179, 'p95': 1.553, 'quantity': 898}, 'triage': {'prom': 0.986, 'p90': 1.318, 'p95': 1.839, 'quantity': 898}, 'router': {'prom': 1.064, 'p90': 1.267, 'p95': 1.943, 'quantity': 898}, 'ag_call': {'prom': 3.47, 'p90': 4.662, 'p95': 6.99, 'quantity': 499}, 'personality': {'prom': 1.605, 'p90': 2.265, 'p95': 2.732, 'quantity': 499}, 'grounding': {'prom': 1.619, 'p90': 2.47, 'p95': 3.0, 'quantity': 499}, 'retriever': {'prom': 0.727, 'p90': 1.172, 'p95': 1.711, 'quantity': 499}, 'ret_embeddings': {'prom': 0.37, 'p90': 0.74, 'p95': 1.284, 'quantity': 499}, 'rag_answer': {'prom': 1.795, 'p90': 2.877, 'p95': 3.378, 'quantity': 499}, 'response_time': {'prom': 8.078, 'p90': 8.543, 'p95': 11.47, 'quantity': 530}}, 'tokens': {'in_ref': {'prom': 803.489, 'total': 721533, 'quantity': 898}, 'out_ref': {'prom': 20.155, 'total': 18099, 'quantity': 898}, 'in_tri': {'prom': 2621.791, 'total': 2354368, 'quantity': 898}, 'out_tri': {'prom': 34.663, 'total': 31127, 'quantity': 898}, 'in_rou': {'prom': 1191.472, 'total': 63148, 'quantity': 53}, 'out_rou': {'prom': 39.226, 'total': 2079, 'quantity': 53}, 'in_per': {'prom': 782.265, 'total': 390350, 'quantity': 499}, 'out_per': {'prom': 123.555, 'total': 61654, 'quantity': 499}, 'in_gro': {'prom': 1723.585, 'total': 860069, 'quantity': 499}, 'out_gro': {'prom': 79.701, 'total': 39771, 'quantity': 499}, 'in_rag': {'prom': 1924.397, 'total': 906391, 'quantity': 471}, 'out_rag': {'prom': 107.185, 'total': 50484, 'quantity': 471}, 'retriever': {'prom': 17.769, 'total': 8369, 'quantity': 471}, 'in_tot': {'prom': 5897.393, 'total': 5295859, 'quantity': 898}, 'out_tot': {'prom': 226.296, 'total': 203214, 'quantity': 898}}}

result_two = {'timestamp': '20260330-202728', 'nodes': {'triage': {'positives': 873, 'total': 901, 'result': 96.89}, 'router': {'positives': 802, 'total': 873, 'result': 91.87}}, 'timings': {'reformulate': {'prom': 0.853, 'p90': 1.077, 'p95': 1.576, 'quantity': 898}, 'triage': {'prom': 0.925, 'p90': 1.158, 'p95': 1.686, 'quantity': 898}, 'router': {'prom': 0.759, 'p90': 1.069, 'p95': 1.511, 'quantity': 898}, 'ag_call': {'prom': 2.776, 'p90': 3.665, 'p95': 4.304, 'quantity': 500}, 'personality': {'prom': 1.375, 'p90': 1.864, 'p95': 2.302, 'quantity': 500}, 'grounding': {'prom': 1.363, 'p90': 1.719, 'p95': 2.198, 'quantity': 500}, 'retriever': {'prom': 0.667, 'p90': 0.872, 'p95': 1.222, 'quantity': 500}, 'ret_embeddings': {'prom': 0.346, 'p90': 0.522, 'p95': 0.876, 'quantity': 500}, 'rag_answer': {'prom': 1.586, 'p90': 2.361, 'p95': 2.764, 'quantity': 500}, 'response_time': {'prom': 6.459, 'p90': 7.145, 'p95': 8.288, 'quantity': 540}}, 'tokens': {'in_ref': {'prom': 803.489, 'total': 721533, 'quantity': 898}, 'out_ref': {'prom': 20.117, 'total': 18065, 'quantity': 898}, 'in_tri': {'prom': 2621.749, 'total': 2354331, 'quantity': 898}, 'out_tri': {'prom': 34.693, 'total': 31154, 'quantity': 898}, 'in_rou': {'prom': 1191.661, 'total': 66733, 'quantity': 56}, 'out_rou': {'prom': 39.0, 'total': 2184, 'quantity': 56}, 'in_per': {'prom': 779.581, 'total': 389011, 'quantity': 499}, 'out_per': {'prom': 122.78, 'total': 61267, 'quantity': 499}, 'in_gro': {'prom': 1722.634, 'total': 861317, 'quantity': 500}, 'out_gro': {'prom': 78.064, 'total': 39032, 'quantity': 500}, 'in_rag': {'prom': 1915.856, 'total': 919611, 'quantity': 480}, 'out_rag': {'prom': 105.204, 'total': 50498, 'quantity': 480}, 'retriever': {'prom': 17.79, 'total': 8539, 'quantity': 480}, 'in_tot': {'prom': 5915.964, 'total': 5312536, 'quantity': 898}, 'out_tot': {'prom': 225.167, 'total': 202200, 'quantity': 898}}}

result_3 = {'timestamp': '20260331-082551', 'nodes': {'triage': {'positives': 868, 'total': 901, 'result': 96.34}, 'router': {'positives': 801, 'total': 868, 'result': 92.28}}, 'timings': {'reformulate': {'prom': 0.94, 'p90': 1.243, 'p95': 1.786, 'quantity': 898}, 'triage': {'prom': 1.058, 'p90': 1.405, 'p95': 2.046, 'quantity': 898}, 'router': {'prom': 0.491, 'p90': 1.13, 'p95': 1.484, 'quantity': 898}, 'ag_call': {'prom': 3.146, 'p90': 4.463, 'p95': 5.156, 'quantity': 497}, 'personality': {'prom': 1.861, 'p90': 2.941, 'p95': 3.579, 'quantity': 497}, 'grounding': {'prom': 1.775, 'p90': 2.599, 'p95': 3.381, 'quantity': 497}, 'retriever': {'prom': 0.609, 'p90': 0.78, 'p95': 1.224, 'quantity': 497}, 'ret_embeddings': {'prom': 0.273, 'p90': 0.447, 'p95': 0.773, 'quantity': 497}, 'rag_answer': {'prom': 1.991, 'p90': 3.064, 'p95': 3.769, 'quantity': 497}, 'response_time': {'prom': 7.194, 'p90': 8.892, 'p95': 10.127, 'quantity': 538}}, 'tokens': {'in_ref': {'prom': 803.489, 'total': 721533, 'quantity': 898}, 'out_ref': {'prom': 20.1, 'total': 18050, 'quantity': 898}, 'in_tri': {'prom': 2621.736, 'total': 2354319, 'quantity': 898}, 'out_tri': {'prom': 34.797, 'total': 31248, 'quantity': 898}, 'in_rou': {'prom': 1191.276, 'total': 69094, 'quantity': 58}, 'out_rou': {'prom': 38.448, 'total': 2230, 'quantity': 58}, 'in_per': {'prom': 783.303, 'total': 387735, 'quantity': 495}, 'out_per': {'prom': 124.796, 'total': 61774, 'quantity': 495}, 'in_gro': {'prom': 1725.907, 'total': 857776, 'quantity': 497}, 'out_gro': {'prom': 79.652, 'total': 39587, 'quantity': 497}, 'in_rag': {'prom': 1933.836, 'total': 897300, 'quantity': 464}, 'out_rag': {'prom': 108.985, 'total': 50569, 'quantity': 464}, 'retriever': {'prom': 17.707, 'total': 8216, 'quantity': 464}, 'in_tot': {'prom': 5888.371, 'total': 5287757, 'quantity': 898}, 'out_tot': {'prom': 226.568, 'total': 203458, 'quantity': 898}}}

result_4 = {'timestamp': '20260331-111651', 'nodes': {'triage': {'positives': 870, 'total': 901, 'result': 96.56}, 'router': {'positives': 803, 'total': 870, 'result': 92.3}}, 'timings': {'reformulate': {'prom': 1.059, 'p90': 1.438, 'p95': 2.123, 'quantity': 898}, 'triage': {'prom': 1.216, 'p90': 1.901, 'p95': 2.474, 'quantity': 898}, 'router': {'prom': 0.667, 'p90': 1.084, 'p95': 1.644, 'quantity': 898}, 'ag_call': {'prom': 3.458, 'p90': 4.842, 'p95': 6.072, 'quantity': 498}, 'personality': {'prom': 2.011, 'p90': 3.171, 'p95': 4.007, 'quantity': 498}, 'grounding': {'prom': 1.829, 'p90': 2.835, 'p95': 3.504, 'quantity': 498}, 'retriever': {'prom': 0.668, 'p90': 0.865, 'p95': 1.361, 'quantity': 498}, 'ret_embeddings': {'prom': 0.306, 'p90': 0.465, 'p95': 0.952, 'quantity': 498}, 'rag_answer': {'prom': 2.117, 'p90': 3.481, 'p95': 4.105, 'quantity': 498}, 'response_time': {'prom': 7.701, 'p90': 9.692, 'p95': 11.747, 'quantity': 586}}, 'tokens': {'in_ref': {'prom': 803.489, 'total': 721533, 'quantity': 898}, 'out_ref': {'prom': 20.108, 'total': 18057, 'quantity': 898}, 'in_tri': {'prom': 2621.735, 'total': 2354318, 'quantity': 898}, 'out_tri': {'prom': 34.726, 'total': 31184, 'quantity': 898}, 'in_rou': {'prom': 1191.661, 'total': 66733, 'quantity': 56}, 'out_rou': {'prom': 38.107, 'total': 2134, 'quantity': 56}, 'in_per': {'prom': 781.072, 'total': 388974, 'quantity': 498}, 'out_per': {'prom': 123.418, 'total': 61462, 'quantity': 498}, 'in_gro': {'prom': 1719.659, 'total': 856390, 'quantity': 498}, 'out_gro': {'prom': 78.735, 'total': 39210, 'quantity': 498}, 'in_rag': {'prom': 1915.171, 'total': 905876, 'quantity': 473}, 'out_rag': {'prom': 106.072, 'total': 50172, 'quantity': 473}, 'retriever': {'prom': 17.638, 'total': 8343, 'quantity': 473}, 'in_tot': {'prom': 5895.127, 'total': 5293824, 'quantity': 898}, 'out_tot': {'prom': 225.188, 'total': 202219, 'quantity': 898}}}

result_5 = {'timestamp': '20260331-130122', 'nodes': {'triage': {'positives': 864, 'total': 901, 'result': 95.89}, 'router': {'positives': 791, 'total': 864, 'result': 91.55}}, 'timings': {'reformulate': {'prom': 0.966, 'p90': 1.339, 'p95': 2.018, 'quantity': 898}, 'triage': {'prom': 1.1, 'p90': 1.647, 'p95': 2.347, 'quantity': 898}, 'router': {'prom': 0.858, 'p90': 1.23, 'p95': 1.934, 'quantity': 898}, 'ag_call': {'prom': 3.111, 'p90': 4.425, 'p95': 5.178, 'quantity': 496}, 'personality': {'prom': 1.87, 'p90': 3.053, 'p95': 3.873, 'quantity': 496}, 'grounding': {'prom': 1.759, 'p90': 2.689, 'p95': 3.351, 'quantity': 496}, 'retriever': {'prom': 0.655, 'p90': 0.866, 'p95': 1.131, 'quantity': 496}, 'ret_embeddings': {'prom': 0.272, 'p90': 0.48, 'p95': 0.749, 'quantity': 496}, 'rag_answer': {'prom': 1.937, 'p90': 2.892, 'p95': 3.735, 'quantity': 496}, 'response_time': {'prom': 7.594, 'p90': 9.121, 'p95': 10.522, 'quantity': 553}}, 'tokens': {'in_ref': {'prom': 803.489, 'total': 721533, 'quantity': 898}, 'out_ref': {'prom': 20.153, 'total': 18097, 'quantity': 898}, 'in_tri': {'prom': 2621.786, 'total': 2354364, 'quantity': 898}, 'out_tri': {'prom': 34.62, 'total': 31089, 'quantity': 898}, 'in_rou': {'prom': 1191.754, 'total': 67930, 'quantity': 57}, 'out_rou': {'prom': 38.86, 'total': 2215, 'quantity': 57}, 'in_per': {'prom': 781.065, 'total': 386627, 'quantity': 495}, 'out_per': {'prom': 122.82, 'total': 60796, 'quantity': 495}, 'in_gro': {'prom': 1720.143, 'total': 853191, 'quantity': 496}, 'out_gro': {'prom': 79.115, 'total': 39241, 'quantity': 496}, 'in_rag': {'prom': 1939.84, 'total': 898146, 'quantity': 463}, 'out_rag': {'prom': 108.225, 'total': 50108, 'quantity': 463}, 'retriever': {'prom': 17.836, 'total': 8258, 'quantity': 463}, 'in_tot': {'prom': 5881.727, 'total': 5281791, 'quantity': 898}, 'out_tot': {'prom': 224.439, 'total': 201546, 'quantity': 898}}}

{'timestamp': '20260406-153646', 'nodes': {'triage': {'positives': 156, 'total': 163, 'result': 95.71}, 'router': {'positives': 152, 'total': 156, 'result': 97.44}, 'grounding': {'positives': 131, 'total': 156, 'result': 83.97}}}


In [ ]:
save_results_on_cosmos(results)